# 03 PySpark ML Fraud Modeling

This notebook builds supervised machine learning models in PySpark to predict fraudulent credit card transactions.

The modeling workflow includes:
- loading the transaction data with Spark
- applying shared cleaning and feature engineering
- training a Logistic Regression baseline
- comparing performance against a Random Forest model
- evaluating tradeoffs between predictive performance and operational speed

In [27]:
#Imports
from pathlib import Path

from pyspark.sql import functions as F

from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.sql.window import Window


from project.config import load_config
from project.spark_session import create_spark
from project.schemas import TRANSACTION_SCHEMA
from project.transform import prepare_transactions

## Load configuration and start Spark

We load the project configuration and start a local Spark session.  
This keeps the notebook aligned with the same reusable project structure used in the EDA and streaming notebooks.

In [3]:
repo_root = Path.cwd().parent
cfg = load_config(repo_root / "configs" / "local.yaml")

raw_csv = repo_root / cfg.paths.raw

spark = create_spark(cfg)
spark.sparkContext.setLogLevel("WARN")

print("Spark session created.")
print(f"Raw data path: {raw_csv}")

Spark session created.
Raw data path: /home/jovyan/work/data/raw/credit_card_transactions.csv


## Read the dataset with Spark

We load the full transaction dataset using the shared transaction schema.  
The same schema is reused across the project for consistency.

In [4]:
df = (
    spark.read
         .option("header", True)
         .option("timestampFormat", "yyyy-MM-dd HH:mm:ss")
         .option("dateFormat", "yyyy-MM-dd")
         .schema(TRANSACTION_SCHEMA)
         .csv(str(raw_csv))
)

In [5]:
df.printSchema()
print(f"Row count: {df.count():,}")

root
 |-- Unnamed: 0: long (nullable = true)
 |-- trans_date_trans_time: timestamp (nullable = true)
 |-- cc_num: string (nullable = true)
 |-- merchant: string (nullable = true)
 |-- category: string (nullable = true)
 |-- amt: double (nullable = true)
 |-- first: string (nullable = true)
 |-- last: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- street: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- zip: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- long: double (nullable = true)
 |-- city_pop: integer (nullable = true)
 |-- job: string (nullable = true)
 |-- dob: date (nullable = true)
 |-- trans_num: string (nullable = true)
 |-- unix_time: long (nullable = true)
 |-- merch_lat: double (nullable = true)
 |-- merch_long: double (nullable = true)
 |-- is_fraud: integer (nullable = true)
 |-- merch_zipcode: string (nullable = true)

Row count: 1,296,675


## Apply reusable data transformations

We apply the shared transformation pipeline from `transform.py` to clean and enrich the dataset.

This step adds modeling-friendly features such as:
- transaction hour
- day of week
- customer age
- log-transformed transaction amount

In [6]:
df = prepare_transactions(df)

In [7]:
df.printSchema()

df.select(
    "amt",
    "category",
    "gender",
    "state",
    "event_hour",
    "event_dayofweek",
    "age",
    "is_fraud"
).show(5, truncate=False)

root
 |-- Unnamed: 0: long (nullable = true)
 |-- trans_date_trans_time: timestamp (nullable = true)
 |-- cc_num: string (nullable = true)
 |-- merchant: string (nullable = true)
 |-- category: string (nullable = true)
 |-- amt: double (nullable = true)
 |-- first: string (nullable = true)
 |-- last: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- street: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- zip: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- long: double (nullable = true)
 |-- city_pop: integer (nullable = true)
 |-- job: string (nullable = true)
 |-- dob: date (nullable = true)
 |-- trans_num: string (nullable = true)
 |-- unix_time: long (nullable = true)
 |-- merch_lat: double (nullable = true)
 |-- merch_long: double (nullable = true)
 |-- is_fraud: integer (nullable = true)
 |-- merch_zipcode: string (nullable = true)
 |-- event_date: date (nullable = true)
 |-- event_hour: int

## Select modeling features

For the first-pass fraud model, we use a focused set of numeric and categorical features that are likely to be predictive while remaining efficient and interpretable.

We avoid identifier columns such as account numbers and transaction IDs in the initial model.

In [8]:
numeric_features = [
    "amt",
    "city_pop",
    "event_hour",
    "event_dayofweek",
    "age",
]

categorical_features = [
    "category",
    "gender",
    "state",
]

label_col = "is_fraud"

In [9]:
model_cols = numeric_features + categorical_features + [label_col]

model_df = df.select(*model_cols).dropna()

print(f"Modeling row count after dropna: {model_df.count():,}")
model_df.groupBy("is_fraud").count().show()

Modeling row count after dropna: 1,296,675
+--------+-------+
|is_fraud|  count|
+--------+-------+
|       1|   7506|
|       0|1289169|
+--------+-------+



## Split the data into training and test sets

We create a train/test split so that model performance can be evaluated on unseen transactions.

In [10]:
train_df, test_df = model_df.randomSplit([0.8, 0.2], seed=42)

print(f"Train rows: {train_df.count():,}")
print(f"Test rows: {test_df.count():,}")

Train rows: 1,037,704
Test rows: 258,971


## Build the PySpark feature pipeline

We encode categorical variables, assemble numeric and encoded categorical features into a single feature vector, and pass that vector to the classifier.

In [11]:
indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
    for c in categorical_features
]

encoders = [
    OneHotEncoder(inputCol=f"{c}_idx", outputCol=f"{c}_ohe")
    for c in categorical_features
]

assembler_inputs = numeric_features + [f"{c}_ohe" for c in categorical_features]

assembler = VectorAssembler(
    inputCols=assembler_inputs,
    outputCol="features"
)

## Train a Logistic Regression baseline

Logistic Regression provides a strong baseline because it is fast, scalable, and relatively interpretable for binary classification problems such as fraud detection.

In [12]:
lr = LogisticRegression(
    featuresCol="features",
    labelCol=label_col,
    predictionCol="prediction",
    probabilityCol="probability",
    rawPredictionCol="rawPrediction",
    maxIter=50,
    regParam=0.0,
    elasticNetParam=0.0
)

lr_pipeline = Pipeline(stages=indexers + encoders + [assembler, lr])

In [13]:
lr_model = lr_pipeline.fit(train_df)
lr_predictions = lr_model.transform(test_df)

In [14]:
lr_predictions.select(
    "is_fraud",
    "prediction",
    "probability"
).show(10, truncate=False)

+--------+----------+------------------------------------------+
|is_fraud|prediction|probability                               |
+--------+----------+------------------------------------------+
|0       |0.0       |[0.9979893763084158,0.0020106236915842235]|
|0       |0.0       |[0.9986473286522206,0.0013526713477793884]|
|0       |0.0       |[0.9982354617046156,0.0017645382953843658]|
|0       |0.0       |[0.999668574413973,3.314255860269455E-4]  |
|0       |0.0       |[0.9973934539889494,0.0026065460110505922]|
|0       |0.0       |[0.9945737158271156,0.005426284172884377] |
|0       |0.0       |[0.9999936883183557,6.311681644333156E-6] |
|0       |0.0       |[0.9992393423629022,7.606576370977614E-4] |
|0       |0.0       |[0.9989658456879473,0.0010341543120526708]|
|0       |0.0       |[0.9977577205053783,0.002242279494621746] |
+--------+----------+------------------------------------------+
only showing top 10 rows


## Evaluate Logistic Regression

Because fraud detection is an imbalanced classification problem, accuracy alone is not sufficient.  
We evaluate the model using AUC, precision, recall, and F1.

In [15]:
binary_eval = BinaryClassificationEvaluator(
    labelCol=label_col,
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

f1_eval = MulticlassClassificationEvaluator(
    labelCol=label_col,
    predictionCol="prediction",
    metricName="f1"
)

precision_eval = MulticlassClassificationEvaluator(
    labelCol=label_col,
    predictionCol="prediction",
    metricName="weightedPrecision"
)

recall_eval = MulticlassClassificationEvaluator(
    labelCol=label_col,
    predictionCol="prediction",
    metricName="weightedRecall"
)

In [16]:
lr_auc = binary_eval.evaluate(lr_predictions)
lr_f1 = f1_eval.evaluate(lr_predictions)
lr_precision = precision_eval.evaluate(lr_predictions)
lr_recall = recall_eval.evaluate(lr_predictions)

print("Logistic Regression Metrics")
print(f"AUC:       {lr_auc:.4f}")
print(f"F1:        {lr_f1:.4f}")
print(f"Precision: {lr_precision:.4f}")
print(f"Recall:    {lr_recall:.4f}")

Logistic Regression Metrics
AUC:       0.8283
F1:        0.9910
Precision: 0.9892
Recall:    0.9937


In [17]:
lr_predictions.groupBy("is_fraud", "prediction").count().orderBy("is_fraud", "prediction").show()

+--------+----------+------+
|is_fraud|prediction| count|
+--------+----------+------+
|       0|       0.0|257330|
|       0|       1.0|    95|
|       1|       0.0|  1527|
|       1|       1.0|    19|
+--------+----------+------+



## Train a Random Forest comparison model

Random Forest can capture nonlinear patterns and interactions that Logistic Regression may miss.  
We compare its predictive performance against the faster, simpler baseline.

In [18]:
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol=label_col,
    predictionCol="prediction",
    probabilityCol="probability",
    rawPredictionCol="rawPrediction",
    numTrees=100,
    maxDepth=10,
    seed=42
)

rf_pipeline = Pipeline(stages=indexers + encoders + [assembler, rf])

In [19]:
rf_model = rf_pipeline.fit(train_df)
rf_predictions = rf_model.transform(test_df)

In [20]:
rf_auc = binary_eval.evaluate(rf_predictions)
rf_f1 = f1_eval.evaluate(rf_predictions)
rf_precision = precision_eval.evaluate(rf_predictions)
rf_recall = recall_eval.evaluate(rf_predictions)

print("Random Forest Metrics")
print(f"AUC:       {rf_auc:.4f}")
print(f"F1:        {rf_f1:.4f}")
print(f"Precision: {rf_precision:.4f}")
print(f"Recall:    {rf_recall:.4f}")

Random Forest Metrics
AUC:       0.9819
F1:        0.9938
Precision: 0.9953
Recall:    0.9953


In [21]:
rf_predictions.groupBy("is_fraud", "prediction").count().orderBy("is_fraud", "prediction").show()

+--------+----------+------+
|is_fraud|prediction| count|
+--------+----------+------+
|       0|       0.0|257425|
|       1|       0.0|  1216|
|       1|       1.0|   330|
+--------+----------+------+



## Compare model performance

We compare the two models to assess the tradeoff between:
- predictive power
- model complexity
- operational suitability for near-real-time fraud scoring

In [35]:
comparison_rows = [
    ("Logistic Regression", round(lr_auc, 6), round(lr_f1, 6), round(lr_precision, 6), round(lr_recall, 6)),
    ("Random Forest", round(rf_auc, 6), round(rf_f1, 6), round(rf_precision, 6), round(rf_recall, 6)),
]

comparison_df = spark.createDataFrame(
    comparison_rows,
    ["model", "auc", "f1", "precision", "recall"]
)

comparison_df.show()

+-------------------+--------+--------+---------+--------+
|              model|     auc|      f1|precision|  recall|
+-------------------+--------+--------+---------+--------+
|Logistic Regression|0.828345|0.991044| 0.989161|0.993737|
|      Random Forest|0.981934|0.993788| 0.995327|0.995304|
+-------------------+--------+--------+---------+--------+



## Model Performance Observation

At first glance, both Logistic Regression and Random Forest models appear to perform extremely well, with high AUC, F1, precision, and recall scores.

However, these metrics are misleading due to the extreme class imbalance in the dataset.

Fraudulent transactions represent only a small fraction of the data. As a result, the models can achieve high overall performance by simply predicting the majority class (non-fraud) for most observations.

This behavior is confirmed by examining prediction outputs and confusion matrices, where the model rarely identifies fraudulent transactions correctly.

Key issue:
- The reported F1, precision, and recall are **weighted metrics**, which are dominated by the majority class (non-fraud)
- As a result, strong performance on non-fraud inflates the overall metrics
- The model still performs poorly at detecting fraud, which is the primary objective

This highlights a fundamental challenge in fraud detection:
- Standard evaluation metrics can be misleading under class imbalance
- Models must be specifically optimized to detect minority class behavior

To address this, the next steps include:
- applying class weighting to emphasize fraud cases
- engineering features that better capture anomalous behavior
- re-evaluating model performance with improved techniques

In [31]:
window = Window.partitionBy()

print("Logistic Regression – Fraud-only performance")

lr_fraud = lr_predictions.filter(F.col("is_fraud") == 1)

lr_fraud_summary = (
    lr_fraud
    .groupBy("prediction")
    .count()
    .withColumn("total_fraud", F.sum("count").over(window))
    .withColumn("percentage", F.col("count") / F.col("total_fraud"))
    .withColumn("percentage", F.round(F.col("percentage") * 100, 2))
    .orderBy("prediction")
)

lr_fraud_summary.show()

Logistic Regression – Fraud-only performance
+----------+-----+-----------+----------+
|prediction|count|total_fraud|percentage|
+----------+-----+-----------+----------+
|       0.0| 1527|       1546|     98.77|
|       1.0|   19|       1546|      1.23|
+----------+-----+-----------+----------+



In [32]:
print("Random Forest – Fraud-only performance")

rf_fraud = rf_predictions.filter(F.col("is_fraud") == 1)

rf_fraud_summary = (
    rf_fraud
    .groupBy("prediction")
    .count()
    .withColumn("total_fraud", F.sum("count").over(window))
    .withColumn("percentage", F.col("count") / F.col("total_fraud"))
    .withColumn("percentage", F.round(F.col("percentage") * 100, 2))
    .orderBy("prediction")
)

rf_fraud_summary.show()

Random Forest – Fraud-only performance
+----------+-----+-----------+----------+
|prediction|count|total_fraud|percentage|
+----------+-----+-----------+----------+
|       0.0| 1216|       1546|     78.65|
|       1.0|  330|       1546|     21.35|
+----------+-----+-----------+----------+



### Fraud Detection Performance

The fraud-only evaluation highlights a critical issue with both models.

- Logistic Regression correctly identifies only **~1%** of fraud cases
- Random Forest improves performance but still captures only **~21%** of fraud cases

This means that the majority of fraudulent transactions are being classified as non-fraud.

Despite strong overall evaluation metrics, both models perform poorly on the actual objective of fraud detection due to class imbalance. The models are biased toward predicting the majority class (non-fraud), which dominates the dataset.

### Next Steps: Improving Fraud Detection

To address these limitations, we will improve the modeling approach using the following strategies:

1. **Class Imbalance Handling**
   - Apply class weighting to penalize misclassification of fraud cases more heavily

2. **Feature Engineering**
   - Introduce features that better capture anomalous behavior, such as:
     - transactions occurring at unusual times (e.g., late night)
     - abnormal transaction amounts
     - geographic distance between customer and merchant

3. **Model Re-evaluation**
   - Retrain models using the improved features and weighting
   - Compare fraud detection performance before and after improvements

These steps aim to increase the model’s ability to detect fraudulent transactions while maintaining scalability for real-world applications.